# Все каналы => пул => ранкер => сабмит

Исторические ноутбуки сохранены. Один split и один генератор признаков для обучения, проверки и benchmark. Метки outer-val не попадают в history ранкера.

In [1]:
from pipeline.pool_experiments import PoolExperiment, FEATURES
experiment = PoolExperiment(".", sample_size=5000)
print("features:", len(FEATURES))

history=448040, eval=5000, items=344825
features: 64


## Общий пул

Weighted BM25 3:1:1, отдельные BM25-поля, строгий локальный BM25, эмбеды, lookup, fuzzy, char TF-IDF, гео и фильтры.\
 Геопризнаки доступны каждому кандидату.\
  Выбор top-500 идёт после общего скоринга.

In [2]:
channel_table = experiment.compare()
display(channel_table.round(6))
print("selected pool:", experiment.pool_variant)

Подготовка каналов: 5000 запросов
каналы: 500/5000
каналы: 1000/5000
каналы: 1500/5000
каналы: 2000/5000
каналы: 2500/5000
каналы: 3000/5000
каналы: 3500/5000
каналы: 4000/5000
каналы: 4500/5000
каналы: 5000/5000


,Recall@50,Recall@500,pool_ceiling,mean_pool_size
variant,,,,
old_score,0.333235,0.648932,0.648932,496.9212
fixed_geo_score,0.477188,0.739201,0.739201,496.9212
weighted_bm25,0.552337,0.790391,0.790391,496.9212
local_bm25,0.559711,0.809984,0.809984,496.9212
plus_fuzzy,0.541128,0.807172,0.807172,497.0364
plus_char,0.532739,0.804395,0.804395,498.2962
all_channels,0.532314,0.804293,0.804293,499.2216
all_geo_weight_0.5,0.750567,0.902925,0.902925,499.2216
all_geo_weight_1,0.815249,0.927362,0.927362,499.2216


selected pool: all_geo_weight_1


## Обучение и одинаковая проверка

Оба ранкера обучаются на внутреннем split истории. Позитивы не добавляются в пул принудительно. Старый контроль использует прежние шесть каналов и признаки.

In [3]:
validation_training = experiment.train_ranker(fit_queries=6000)
print(validation_training)

validation: готово 0, осталось 6000 контекстов
Подготовка каналов: 6000 запросов
каналы: 500/6000
каналы: 1000/6000
каналы: 1500/6000
каналы: 2000/6000
каналы: 2500/6000
каналы: 3000/6000
каналы: 3500/6000
каналы: 4000/6000
каналы: 4500/6000
каналы: 5000/6000
каналы: 5500/6000
каналы: 6000/6000
0:	learn: 0.4978702	test: 0.5003238	best: 0.5003238 (0)	total: 727ms	remaining: 3m 1s
50:	learn: 0.0595498	test: 0.0701852	best: 0.0701852 (50)	total: 32.5s	remaining: 2m 6s
100:	learn: 0.0504243	test: 0.0667369	best: 0.0661831 (94)	total: 1m 3s	remaining: 1m 33s
150:	learn: 0.0445391	test: 0.0665724	best: 0.0661064 (126)	total: 1m 33s	remaining: 1m 1s
Stopped by overfitting detector  (40 iterations wait)

bestTest = 0.06610643311
bestIteration = 126

Shrink model to first 127 iterations.
0:	learn: 0.5066237	test: 0.5083854	best: 0.5083854 (0)	total: 278ms	remaining: 1m 9s
50:	learn: 0.0414410	test: 0.0516239	best: 0.0516239 (50)	total: 13.4s	remaining: 52.2s
100:	learn: 0.0335529	test: 0.049429

In [4]:
validation_table, warm_cold, context_counts = experiment.evaluate_ranker()
display(validation_table.round(6))
display(warm_cold.round(6))
print(context_counts)

,Recall@50,Recall@500,pool_ceiling,mean_pool_size
variant,,,,
old_score,0.333235,0.648932,0.648932,496.9212
fixed_geo_score,0.477188,0.739201,0.739201,496.9212
weighted_bm25,0.552337,0.790391,0.790391,496.9212
local_bm25,0.559711,0.809984,0.809984,496.9212
plus_fuzzy,0.541128,0.807172,0.807172,497.0364
plus_char,0.532739,0.804395,0.804395,498.2962
all_channels,0.532314,0.804293,0.804293,499.2216
all_geo_weight_0.5,0.750567,0.902925,0.902925,499.2216
all_geo_weight_1,0.815249,0.927362,0.927362,499.2216


,warm_recall50,cold_recall50
variant,,
old_score,0.321458,0.414046
fixed_geo_score,0.468478,0.536950
weighted_bm25,0.548164,0.580975
local_bm25,0.556841,0.579403
plus_fuzzy,0.537039,0.569182
plus_char,0.526130,0.578092
all_channels,0.525184,0.581237
all_geo_weight_0.5,0.752445,0.737683
all_geo_weight_1,0.821207,0.774371


{'warm_contexts': 4364, 'cold_contexts': 636}

## Проверка на бенче

Берём outer-val контексты с известным релевантным item в benchmark.\
 Recall условный: в знаменателе только такие item. Это proxy, а не скор платформы. 70% контекстов выбирают политику, 30% проверяют её; отдельно warm/cold. Итоговая смесь приближает долю новых текстов benchmark: 63%.

In [5]:
benchmark_table = experiment.evaluate_benchmark_proxy()
display(benchmark_table.round(6))
print("submission policy:", experiment.final_policy)

benchmark proxy: 2679 контекстов, только релевантные item benchmark
Подготовка каналов: 2679 запросов
каналы: 500/2679
каналы: 1000/2679
каналы: 1500/2679
каналы: 2000/2679
каналы: 2500/2679
Политика для submission: ranker_heuristic_geo_rrf_2

,subset,variant,contexts,cold_contexts,Recall@50,Recall@500,warm_recall50,cold_recall50,benchmark_mix_recall50
0,all,updated_catboost,2679,346,0.778676,0.968583,0.788929,0.709538,0.738912
1,all,weighted_geo50,2679,346,0.832564,0.832564,0.837737,0.797688,0.812506
2,all,pool_heuristic,2679,346,0.867107,0.968583,0.876545,0.803468,0.830506
3,all,old_catboost,2679,346,0.641380,0.762895,0.641131,0.643064,0.642348
4,all,ranker_geo_rrf_0.25,2679,346,0.820172,0.968210,0.829078,0.760116,0.785632
5,all,ranker_heuristic_rrf_0.25,2679,346,0.810342,0.968583,0.820577,0.741329,0.770651
6,all,ranker_heuristic_geo_rrf_0.25,2679,346,0.846889,0.968583,0.854186,0.797688,0.818592
7,all,ranker_geo_rrf_0.5,2679,346,0.835475,0.968210,0.843436,0.781792,0.804600
8,all,ranker_heuristic_rrf_0.5,2679,346,0.827776,0.968583,0.838882,0.752890,0.784707
9,all,ranker_heuristic_geo_rrf_0.5,2679,346,0.851647,0.968583,0.859221,0.800578,0.822276


submission policy: ranker_heuristic_geo_rrf_2

## Финальная модель и CSV

Повторяем внутренний split всего train. Статистики submission строятся на всём train, item-корпус — benchmark. Проверяем ровно 50 уникальных допустимых item на каждый query_id. v1 не перезаписывается.

In [6]:
production_training = experiment.train_ranker(fit_queries=8000, production=True)
print(production_training)

production: готово 0, осталось 8000 контекстов
Подготовка каналов: 8000 запросов
каналы: 500/8000
каналы: 1000/8000
каналы: 1500/8000
каналы: 2000/8000
каналы: 2500/8000
каналы: 3000/8000
каналы: 3500/8000
каналы: 4000/8000
каналы: 4500/8000
каналы: 5000/8000
каналы: 5500/8000
каналы: 6000/8000
каналы: 6500/8000
каналы: 7000/8000
каналы: 7500/8000
каналы: 8000/8000
0:	learn: 0.4732600	test: 0.4698563	best: 0.4698563 (0)	total: 956ms	remaining: 3m 58s
50:	learn: 0.0486988	test: 0.0486091	best: 0.0486091 (50)	total: 42.8s	remaining: 2m 47s
100:	learn: 0.0403964	test: 0.0452725	best: 0.0447705 (91)	total: 1m 23s	remaining: 2m 3s
Stopped by overfitting detector  (40 iterations wait)

bestTest = 0.04477053491
bestIteration = 91

Shrink model to first 92 iterations.
0:	learn: 0.4496782	test: 0.4484465	best: 0.4484465 (0)	total: 367ms	remaining: 1m 31s
50:	learn: 0.0354710	test: 0.0405317	best: 0.0405317 (50)	total: 17.9s	remaining: 1m 9s
100:	learn: 0.0287977	test: 0.0387563	best: 0.0386199 

In [7]:
submission_metadata = experiment.make_submission()
print(submission_metadata)

Подготовка каналов: 2452 запросов
каналы: 500/2452
каналы: 1000/2452
каналы: 1500/2452
каналы: 2000/2452

{'file': 'submissions/v3.csv', 'queries': 2452, 'items_per_query': 50, 'pool_variant': 'all_geo_weight_1', 'final_policy': 'ranker_heuristic_geo_rrf_2', 'features': ['bm25_norm', 'geo_distance_km', 'geo_bonus', 'geo_exact', 'vid_match', 'tip_match', 'log_popularity', 'fuzzy_score', 'char_score', 'dense_score', 'title_coverage', 'item_price', 'item_rating', 'log_reviews', 'subcat_match', 'geo_exact_match', 'vid_uslugi_match', 'tip_uslugi_match', 'item_rating_reviews_count', 'item_is_phone_hidden', 'item_is_message_forbidden', 'item_popularity', 'text_overlap', 'bm25_rank', 'bm25_present', 'bm25_score', 'dense_rank', 'dense_present', 'lookup_rank', 'lookup_present', 'lookup_score', 'geo_rank', 'geo_present', 'geo_score', 'filter_rank', 'filter_present', 'filter_score', 'geo_exact_rank', 'geo_exact_present', 'geo_exact_score', 'weighted_rank', 'weighted_present', 'weighted_score', 'local_rank', 'local_present', 'local_score', 'fuzzy_rank', 'fuzzy_present', 'char_rank', 'char_present', 'ti

## Результат

Финальный CSV: `submissions/v<n>.csv`, 2452 запроса по 50 уникальных item. Модель: 205 деревьев, 64 признака. Смесь ранкер/эвристика/гео = 1/2/0.5 выбрана на benchmark-proxy. Метрики ниже получены на validation-модели, production-модель обучена на внутреннем split всего train. Benchmark-proxy учитывает только известные релевантные item этого каталога; это не скор платформы.


In [8]:
import pandas as pd
from catboost import CatBoostRanker
from pipeline.pool_experiments import FEATURES

old_scores = pd.read_csv("data/pool_eval/final_comparison.csv", index_col=0)
proxy = pd.read_csv("data/pool_eval/benchmark_proxy.csv")
holdout = proxy.loc[proxy["subset"] == "policy_holdout"].set_index("variant")
summary = pd.DataFrame([
    {"comparison": "Recall@50: 5000 контекстов, ranker vs legacy control", "old": old_scores.loc["old_control_catboost", "Recall@50"], "new": old_scores.loc["updated_catboost", "Recall@50"]},
    {"comparison": "Recall@50: benchmark proxy holdout, итоговая политика vs legacy", "old": holdout.loc["old_catboost", "Recall@50"], "new": holdout.loc[experiment.final_policy, "Recall@50"]},
    {"comparison": "Pool ceiling: 5000 контекстов", "old": old_scores.loc["old_control_catboost", "pool_ceiling"], "new": old_scores.loc["updated_catboost", "pool_ceiling"]},
])
display(summary.round(6))
model = CatBoostRanker()
model.load_model("data/pool_eval/production_ranker.cbm")
importance = pd.DataFrame({"feature": FEATURES, "importance": model.get_feature_importance(type="PredictionValuesChange")}).sort_values("importance", ascending=False)
importance.to_csv("data/pool_eval/production_feature_importance.csv", index=False)
display(importance.head(12).round(4))
print(submission_metadata)


,comparison,old,new
0,"Recall@50: 5000 контекстов, ranker vs legacy c...",0.512413,0.882367
1,"Recall@50: benchmark proxy holdout, итоговая п...",0.629270,0.855100
2,Pool ceiling: 5000 контекстов,0.533668,0.927362


,feature,importance
1,geo_distance_km,27.0165
2,geo_bonus,13.4172
0,bm25_norm,6.7464
16,vid_uslugi_match,5.9942
59,title_norm,4.2242
43,local_rank,3.0581
44,local_present,2.9246
21,item_popularity,2.9177
40,weighted_rank,2.6950
11,item_price,2.6543


{'file': 'submissions/v3.csv', 'queries': 2452, 'items_per_query': 50, 'pool_variant': 'all_geo_weight_1', 'final_policy': 'ranker_heuristic_geo_rrf_2', 'features': ['bm25_norm', 'geo_distance_km', 'geo_bonus', 'geo_exact', 'vid_match', 'tip_match', 'log_popularity', 'fuzzy_score', 'char_score', 'dense_score', 'title_coverage', 'item_price', 'item_rating', 'log_reviews', 'subcat_match', 'geo_exact_match', 'vid_uslugi_match', 'tip_uslugi_match', 'item_rating_reviews_count', 'item_is_phone_hidden', 'item_is_message_forbidden', 'item_popularity', 'text_overlap', 'bm25_rank', 'bm25_present', 'bm25_score', 'dense_rank', 'dense_present', 'lookup_rank', 'lookup_present', 'lookup_score', 'geo_rank', 'geo_present', 'geo_score', 'filter_rank', 'filter_present', 'filter_score', 'geo_exact_rank', 'geo_exact_present', 'geo_exact_score', 'weighted_rank', 'weighted_present', 'weighted_score', 'local_rank', 'local_present', 'local_score', 'fuzzy_rank', 'fuzzy_present', 'char_rank', 'char_present', 'ti